# Stage 1 GNN graph-screening survival plots

This notebook visualizes the Bus14 Stage 1 representation × normalization screening defined in `configs/gnn_graph_screening/stage1_representation_normalization`. It compares three graph representations and four normalization choices using test episodic survival.

The notebook reads the run list from `manifest.csv`, loads cached W&B histories by default, excludes non-canonical reruns, and writes interactive HTML figures to `Topology_Task/outputs/wandb_figures`.

In [1]:
from pathlib import Path
import importlib
import sys

import pandas as pd
import plotly.express as px

for candidate in [Path.cwd(), *Path.cwd().parents]:
    helpers = candidate / "helpers"
    repo_helpers = candidate / "Topology_Task" / "analysis" / "metrics" / "helpers"
    if helpers.exists() and (helpers / "wandb_metrics.py").exists():
        sys.path.insert(0, str(helpers))
        break
    if repo_helpers.exists() and (repo_helpers / "wandb_metrics.py").exists():
        sys.path.insert(0, str(repo_helpers))
        break
else:
    raise FileNotFoundError("Could not locate Topology_Task/analysis/metrics/helpers")

import wandb_metrics as wm
wm = importlib.reload(wm)

CONFIG_DIR = wm.TASK_DIR / "configs" / "gnn_graph_screening" / "stage1_representation_normalization"
MANIFEST_PATH = CONFIG_DIR / "manifest.csv"
print("wandb_metrics:", wm.__file__)
print("screening configs:", CONFIG_DIR)

wandb_metrics: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/analysis/metrics/helpers/wandb_metrics.py
screening configs: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/configs/gnn_graph_screening/stage1_representation_normalization


## Screening design

The screen contains 12 single-seed runs: `bus`, `heterogeneous`, and `heterogeneous_line`, each tested with no graph-feature normalization, physical scaling only, running normalization only, and both methods. All other training choices are held fixed by the Stage 1 configs.

In [2]:
REPRESENTATION_ORDER = ["bus", "heterogeneous", "heterogeneous_line"]
NORMALIZATION_ORDER = ["n0_none", "n1_physical", "n2_running", "n3_both"]

REPRESENTATION_LABELS = {
    "bus": "Bus graph",
    "heterogeneous": "Heterogeneous graph",
    "heterogeneous_line": "Heterogeneous + line nodes",
}
NORMALIZATION_LABELS = {
    "n0_none": "None",
    "n1_physical": "Physical scaling",
    "n2_running": "Running normalization",
    "n3_both": "Physical + running",
}

screen = pd.read_csv(MANIFEST_PATH)
screen["run_name"] = screen["config"].map(lambda name: Path(name).stem)
screen["representation"] = screen["graph_type"].map(REPRESENTATION_LABELS)
screen["normalization_label"] = screen["normalization"].map(NORMALIZATION_LABELS)
screen["graph_type"] = pd.Categorical(screen["graph_type"], REPRESENTATION_ORDER, ordered=True)
screen["normalization"] = pd.Categorical(screen["normalization"], NORMALIZATION_ORDER, ordered=True)
screen = screen.sort_values(["graph_type", "normalization"]).reset_index(drop=True)

display(screen[[
    "run_name", "representation", "normalization_label",
    "physical_scaling", "running_normalization", "seed", "timesteps",
]])

,run_name,representation,normalization_label,physical_scaling,running_normalization,seed,timesteps
0,gs_s1_bus_n0_none_s0,Bus graph,None,False,False,0,8000000
1,gs_s1_bus_n1_physical_s0,Bus graph,Physical scaling,True,False,0,8000000
2,gs_s1_bus_n2_running_s0,Bus graph,Running normalization,False,True,0,8000000
3,gs_s1_bus_n3_both_s0,Bus graph,Physical + running,True,True,0,8000000
4,gs_s1_hetero_n0_none_s0,Heterogeneous graph,None,False,False,0,8000000
5,gs_s1_hetero_n1_physical_s0,Heterogeneous graph,Physical scaling,True,False,0,8000000
6,gs_s1_hetero_n2_running_s0,Heterogeneous graph,Running normalization,False,True,0,8000000
7,gs_s1_hetero_n3_both_s0,Heterogeneous graph,Physical + running,True,True,0,8000000
8,gs_s1_hetero_line_n0_none_s0,Heterogeneous + line nodes,None,False,False,0,8000000
9,gs_s1_hetero_line_n1_physical_s0,Heterogeneous + line nodes,Physical scaling,True,False,0,8000000


## Load the selected histories

`USE_LOCAL_CACHE_ONLY = True` makes the notebook reproducible from downloaded histories and avoids a W&B network query. Set it to `False` only when you want to refresh the runs from W&B. The exact config stems are retained after loading; this deliberately removes three older `hetero_line` attempts whose display names end in `s00`.

In [3]:
USE_LOCAL_CACHE_ONLY = True

wm.configure_run_filter_from_config_folder(CONFIG_DIR)
data = wm.load_wandb_data(use_local_cache_only=USE_LOCAL_CACHE_ONLY)

requested_run_names = screen["run_name"].tolist()
requested_run_name_set = set(requested_run_names)
raw_run_names = set(data.runs_df["name"].astype(str).str.strip())

runs_df = data.runs_df[
    data.runs_df["name"].astype(str).str.strip().isin(requested_run_name_set)
].copy()
history_df = data.history_df[
    data.history_df["run_name"].astype(str).str.strip().isin(requested_run_name_set)
].copy()

found_run_names = set(history_df["run_name"].unique())
missing_run_names = sorted(requested_run_name_set - found_run_names)
ignored_run_names = sorted(raw_run_names - requested_run_name_set)

print(f"Loaded {len(found_run_names)} / {len(requested_run_names)} canonical Stage 1 histories.")
if missing_run_names:
    print("Missing canonical runs:", missing_run_names)
if ignored_run_names:
    print("Ignored non-canonical matching runs:", ignored_run_names)
if history_df.empty:
    raise RuntimeError("No Stage 1 histories were found. Download the runs or disable local-only mode.")

observed_progress = (
    history_df.groupby("run_name", as_index=False)["step"]
    .max()
    .rename(columns={"step": "observed_steps"})
)
run_inventory = screen.merge(observed_progress, on="run_name", how="left")
run_inventory["observed_steps_m"] = run_inventory["observed_steps"] / 1_000_000
run_inventory["configured_budget_m"] = run_inventory["timesteps"] / 1_000_000
run_inventory["observed_budget_pct"] = 100 * run_inventory["observed_steps"] / run_inventory["timesteps"]
display(run_inventory[[
    "run_name", "representation", "normalization_label",
    "observed_steps_m", "configured_budget_m", "observed_budget_pct",
]].round(2))

Config folder filter: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/configs/gnn_graph_screening/stage1_representation_normalization
Matched config run-name candidates: 12
Project: corentin-plumet-epfl/Grid2Op
Task dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task
Cache mode: full
Cache dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_cache
Local-only mode: True
Force refresh: False
Refresh scan-history fallbacks: False
Selected 15 cached runs from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_cache/full_history
state
finished    15
History artifact setup: local_only=True, runs_df=15
[ 1/15] loading artifact cache: gs_s1_bus_n0_none_s0
    loaded 110 rows, 113 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/gs/runs/gs_s1_bus_n0_none_s0__MAPPO_bus14_T_0_0__I__1784700280_36275/history.parquet in 0.1s
[ 2/15] loading artifact cache: gs_s1_bus_n1_physical_s0
    loaded

,run_name,representation,normalization_label,observed_steps_m,configured_budget_m,observed_budget_pct
0,gs_s1_bus_n0_none_s0,Bus graph,None,4.56,8.0,57.02
1,gs_s1_bus_n1_physical_s0,Bus graph,Physical scaling,4.98,8.0,62.21
2,gs_s1_bus_n2_running_s0,Bus graph,Running normalization,5.47,8.0,68.43
3,gs_s1_bus_n3_both_s0,Bus graph,Physical + running,4.89,8.0,61.17
4,gs_s1_hetero_n0_none_s0,Heterogeneous graph,None,4.94,8.0,61.69
5,gs_s1_hetero_n1_physical_s0,Heterogeneous graph,Physical scaling,4.15,8.0,51.84
6,gs_s1_hetero_n2_running_s0,Heterogeneous graph,Running normalization,4.06,8.0,50.80
7,gs_s1_hetero_n3_both_s0,Heterogeneous graph,Physical + running,3.73,8.0,46.66
8,gs_s1_hetero_line_n0_none_s0,Heterogeneous + line nodes,None,3.32,8.0,41.47
9,gs_s1_hetero_line_n1_physical_s0,Heterogeneous + line nodes,Physical scaling,4.06,8.0,50.80


## Plot controls

Curves use a five-evaluation trailing mean. In the all-runs plot, color identifies normalization and dash identifies graph representation. The grouped plots use one visual channel at a time for easier comparison.

In [4]:
SMOOTH_WINDOW = 5
Y_RANGE = [0, 105]
SHOW_RAW = False

NORMALIZATION_COLORS = {
    "n0_none": "#7f7f7f",
    "n1_physical": "#1f77b4",
    "n2_running": "#ff7f0e",
    "n3_both": "#2ca02c",
}
REPRESENTATION_COLORS = {
    "bus": "#1f77b4",
    "heterogeneous": "#ff7f0e",
    "heterogeneous_line": "#2ca02c",
}
REPRESENTATION_DASHES = {
    "bus": "solid",
    "heterogeneous": "dash",
    "heterogeneous_line": "dot",
}

def selected_run_name(graph_type, normalization):
    match = screen[(screen["graph_type"] == graph_type) & (screen["normalization"] == normalization)]
    if len(match) != 1:
        raise RuntimeError(f"Expected one run for {graph_type=} and {normalization=}, found {len(match)}")
    return match.iloc[0]["run_name"]

In [5]:
all_run_specs = [
    {
        "name": selected_run_name(graph_type, normalization),
        "label": f"{REPRESENTATION_LABELS[graph_type]} · {NORMALIZATION_LABELS[normalization]}",
        "color": NORMALIZATION_COLORS[normalization],
        "dash": REPRESENTATION_DASHES[graph_type],
    }
    for graph_type in REPRESENTATION_ORDER
    for normalization in NORMALIZATION_ORDER
]

all_runs_fig = wm.plot_runs(
    all_run_specs,
    split="test",
    smooth=SMOOTH_WINDOW,
    show_raw=SHOW_RAW,
    title="Stage 1: all graph representation × normalization runs",
    y_range=Y_RANGE,
    width=1500,
    height=650,
    save_name="gs_stage1_all_representation_normalization_curves",
    history=history_df,
)
all_runs_fig

Plot source folders used to build curves: 1 folder(s), 12 run(s)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/gs (12 runs)
Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/gs_stage1_all_representation_normalization_curves.html


In [6]:
normalization_within_representation = {
    REPRESENTATION_LABELS[graph_type]: [
        {
            "name": selected_run_name(graph_type, normalization),
            "label": NORMALIZATION_LABELS[normalization],
            "color": NORMALIZATION_COLORS[normalization],
        }
        for normalization in NORMALIZATION_ORDER
    ]
    for graph_type in REPRESENTATION_ORDER
}

normalization_comparison_fig = wm.plot_run_groups(
    normalization_within_representation,
    split="test",
    smooth=SMOOTH_WINDOW,
    show_raw=SHOW_RAW,
    title="Stage 1: normalization comparison within each representation",
    y_range=Y_RANGE,
    ncols=2,
    subplot_height=460,
    width=1500,
    save_name="gs_stage1_normalization_within_representation",
    history=history_df,
)
normalization_comparison_fig

Plot source folders used to build curves: 1 folder(s), 12 run(s)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/gs (12 runs)
Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/gs_stage1_normalization_within_representation.html


In [7]:
representation_within_normalization = {
    NORMALIZATION_LABELS[normalization]: [
        {
            "name": selected_run_name(graph_type, normalization),
            "label": REPRESENTATION_LABELS[graph_type],
            "color": REPRESENTATION_COLORS[graph_type],
        }
        for graph_type in REPRESENTATION_ORDER
    ]
    for normalization in NORMALIZATION_ORDER
}

representation_comparison_fig = wm.plot_run_groups(
    representation_within_normalization,
    split="test",
    smooth=SMOOTH_WINDOW,
    show_raw=SHOW_RAW,
    title="Stage 1: graph-representation comparison within each normalization",
    y_range=Y_RANGE,
    ncols=2,
    subplot_height=460,
    width=1500,
    save_name="gs_stage1_representation_within_normalization",
    history=history_df,
)
representation_comparison_fig

Plot source folders used to build curves: 1 folder(s), 12 run(s)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/gs (12 runs)
Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/gs_stage1_representation_within_normalization.html


## Budget-aligned screening summary

Runs currently contain different numbers of environment steps, so their latest values are not directly comparable. The table reports both the latest five-evaluation mean and a fair value at the largest step reached by every canonical run. The heatmap uses only this common-budget value. A final Stage 1 selection should wait until all runs reach the configured 8M-step budget.

In [8]:
TEST_METRICS = wm.SURVIVAL_METRIC_CANDIDATES["test"]

def survival_frame(run_name):
    run_history = history_df[history_df["run_name"] == run_name]
    for metric in TEST_METRICS:
        frame = run_history[run_history["metric"] == metric][["step", "value"]].copy()
        if frame.empty:
            continue
        frame["step"] = pd.to_numeric(frame["step"], errors="coerce")
        frame["value"] = pd.to_numeric(frame["value"], errors="coerce")
        frame = frame.dropna().groupby("step", as_index=False)["value"].mean().sort_values("step")
        frame["smoothed"] = frame["value"].rolling(SMOOTH_WINDOW, min_periods=1).mean()
        return frame, metric
    return pd.DataFrame(), None

survival_frames = {}
chosen_metrics = {}
for run_name in requested_run_names:
    frame, metric = survival_frame(run_name)
    if not frame.empty:
        survival_frames[run_name] = frame
        chosen_metrics[run_name] = metric

missing_survival = sorted(requested_run_name_set - set(survival_frames))
if missing_survival:
    raise RuntimeError(f"Missing test episodic-survival histories: {missing_survival}")

common_step = min(frame["step"].max() for frame in survival_frames.values())
summary_rows = []
for run_name, frame in survival_frames.items():
    common_frame = frame[frame["step"] <= common_step]
    summary_rows.append({
        "run_name": run_name,
        "metric": chosen_metrics[run_name],
        "last_survival_step_m": frame["step"].iloc[-1] / 1_000_000,
        "latest_5eval_survival_pct": 100 * frame["smoothed"].iloc[-1],
        "best_5eval_survival_pct": 100 * frame["smoothed"].max(),
        "common_budget_survival_pct": 100 * common_frame["smoothed"].iloc[-1],
    })

screening_summary = screen.merge(pd.DataFrame(summary_rows), on="run_name", how="left")
print(f"Common comparison budget: {common_step / 1_000_000:.3f}M environment steps")
display(screening_summary[[
    "representation", "normalization_label", "last_survival_step_m",
    "common_budget_survival_pct", "latest_5eval_survival_pct",
    "best_5eval_survival_pct", "run_name",
]].round(2))

heatmap_data = (
    screening_summary.pivot(
        index="representation",
        columns="normalization_label",
        values="common_budget_survival_pct",
    )
    .reindex([REPRESENTATION_LABELS[key] for key in REPRESENTATION_ORDER])
    .reindex(columns=[NORMALIZATION_LABELS[key] for key in NORMALIZATION_ORDER])
)

common_budget_heatmap = px.imshow(
    heatmap_data,
    text_auto=".1f",
    aspect="auto",
    zmin=0,
    zmax=100,
    color_continuous_scale="Viridis",
    labels={"x": "Normalization", "y": "Graph representation", "color": "Survival (%)"},
    title=f"Stage 1 test survival at the common {common_step / 1_000_000:.3f}M-step budget",
)
common_budget_heatmap.update_layout(width=1050, height=520)
wm.save_plot(common_budget_heatmap, "gs_stage1_common_budget_survival_heatmap")
common_budget_heatmap

Common comparison budget: 3.152M environment steps


,representation,normalization_label,last_survival_step_m,common_budget_survival_pct,latest_5eval_survival_pct,best_5eval_survival_pct,run_name
0,Bus graph,None,4.56,84.50,89.85,90.47,gs_s1_bus_n0_none_s0
1,Bus graph,Physical scaling,4.98,60.85,89.55,89.55,gs_s1_bus_n1_physical_s0
2,Bus graph,Running normalization,5.47,48.57,70.48,73.43,gs_s1_bus_n2_running_s0
3,Bus graph,Physical + running,4.89,43.70,83.82,87.93,gs_s1_bus_n3_both_s0
4,Heterogeneous graph,None,4.89,21.57,88.58,89.05,gs_s1_hetero_n0_none_s0
5,Heterogeneous graph,Physical scaling,4.15,57.52,50.11,61.77,gs_s1_hetero_n1_physical_s0
6,Heterogeneous graph,Running normalization,4.06,52.64,60.33,72.84,gs_s1_hetero_n2_running_s0
7,Heterogeneous graph,Physical + running,3.73,59.85,61.45,68.05,gs_s1_hetero_n3_both_s0
8,Heterogeneous + line nodes,None,3.32,70.12,74.61,77.45,gs_s1_hetero_line_n0_none_s0
9,Heterogeneous + line nodes,Physical scaling,4.06,52.53,43.46,57.00,gs_s1_hetero_line_n1_physical_s0


Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/gs_stage1_common_budget_survival_heatmap.html
